## BMLL Notebook Educational Series
### Venue Ranking
BMLL collects exchange data from over 100 exchanges across the globe. With this core market data, we rebuild Level 1 (top of book), Level 2 (or Market by Price) and Level 3 (or Market by Order) views. We also provide lit and dark trades.  
  
BMLL also calculates derived daily metrics including venue quality metrics. These metrics can be used to create a composite "rank" for venues on a single name basis or across a universe.

This notebook illustrates the methods to calculate a venue ranking score in Snowflake.

In [1]:
import os
from functools import lru_cache
import sys

def install_snowflake_connector():
    os.system(sys.executable + " -m pip install snowflake-connector-python[pandas] snowflake-sqlalchemy")

@lru_cache(5)
def snowflake_connection(
    database="PROD_DERIVED_DATA_DB",
    schema="PUBLIC",
    warehouse="PROD_DERIVED_DEFAULT_WH",
    role="PROD_SALES_RO_ROLE",
    user=None,
    password=None,
    use_secret_storage=True,
    sso=None,
):
    import snowflake
    import snowflake.connector
    try:
        from snowflake.snowpark import Session as SnowparkSession
    except Exception:
        SnowparkSession = None
    try:
        import bmll2
        stor = bmll2.storage_paths()["user"]
        os.environ["SNOWFLAKE_USER"] = stor["prefix"].split("/")[1]
        os.environ["SNOWFLAKE_USE_SSO"] = "1"
    except Exception:
        pass
    if sso is None:
        sso = os.environ.get("SNOWFLAKE_USE_SSO")

    password = os.environ.get("SNOWFLAKE_PASSWORD", password)
    if not password:
        try:
            import secretstorage
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            for item in collection.search_items({"name": "default_snowflake", "system": "bmll"}):
                password = item.get_secret()
        except Exception as e:
            print(f"Error connecting to Secret Service: {e}")
    else:
        try:
            import secretstorage
            attributes = {"name": "default_snowflake", "system": "bmll"}
            label = "default_snowflake"
            secret = password
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            if collection.is_locked():
                unlocked = collection.unlock()
                if unlocked:
                    print("Collection unlocked successfully.")
                else:
                    print("Failed to unlock the collection.")
            else:
                print("Collection is already unlocked.")
            collection.create_item(label, attributes, secret, replace=True)
        except Exception as e:
            print(e)

    if not sso:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            password=password,
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
        )
    else:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
            authenticator="externalbrowser",
        )
    if SnowparkSession is not None:
        return SnowparkSession.builder.configs(connargs).create()
    return snowflake.connector.connect(**connargs)

session = snowflake_connection()
if not hasattr(session, "sql"):
    class _SFResult:
        def __init__(self, cursor):
            self._cursor = cursor

        def to_pandas(self):
            try:
                return self._cursor.fetch_pandas_all()
            except Exception:
                import pandas as pd
                rows = self._cursor.fetchall()
                cols = [c[0] for c in self._cursor.description] if self._cursor.description else []
                return pd.DataFrame(rows, columns=cols)

    class _SFSessionShim:
        def __init__(self, conn):
            self._conn = conn

        def sql(self, query):
            cur = self._conn.cursor()
            cur.execute(query)
            return _SFResult(cur)

    session = _SFSessionShim(session)

/home/bertrand/miniconda3/lib/python3.11/site-packages/bmll2/_internals/api/configure.py:116: UserWarning: BMLL configuration file (/tmp/.nb_extension_instance_vars.json) not found. BMLL data and S3 storage will not be reachable.
  warnings.warn(f"{msg} BMLL data and S3 storage will not be reachable.", UserWarning)


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://accounts.google.com/o/saml2/idp?idpid=C0433grw8&SAMLRequest=lZJdb9owFIb%2FSuRdJ3YCtGAREIOxZaIdKqEXvTPJIXh17MwfpN2vn8OH1F20Ui8SRc77%2BjnnvGc8falFcARtuJIpiiOCApCFKrmsUrTNl%2BEQBcYyWTKhJKToFQyaTsaG1aKhM2cP8gH%2BODA28BdJQ7sfKXJaUsUMN1SyGgy1Bd3M7lY0iQhlxoC2HocultJwzzpY21CM27aN2l6kdIUTQggmI%2BxVneQLeoNoPmY0WllVKHG1vPie3kHEmPQ7hFd4wvpi%2FMrleQQfUXZnkaE%2F8nwdrn9tchTMrt3NlTSuBr0BfeQFbB9W5wKMr2BXCxF2LwvFITJStXvBnqFQdeOsvzHyX3gPJRaq4n5O2SJFzTMvb22mYbQQ3%2Fh%2BN%2Ft9096P1u5vtvteHxPLhz9HWX%2F16Fx%2FmfNtgYLHa6pJl2pmjINMdllaf0SSm5AMQ3KbxwPaG9L%2BIBqQ3hMKFj5LLpk9Oa8Fs6JQTloTVUpVAk71KdwFkWBeNlP%2F8DKdk36vV%2Bl2iM4LQk9MPflM22P81nnZs3s%2F%2BmyxVoIXr8FS6ZrZ95OJo%2Fh0wstwf5JSqBkXs7LUYIxPSAjVzjUw69fZagcIT87U%2Fxd68g8%3D&RelayState=ver%3A3-hint%3A10767547

In [2]:
ticker = 'TSLA'
start_date = '2023-01-01'
end_date = '2023-06-21'

## Import libraries

In [3]:
import pandas as pd

In [4]:
# Expect an existing Snowpark session named `session` in Snowflake worksheets.

## Part 1: Venue Ranking Algorithm & Framework

### Overview
Exchange venues vary in their execution quality and liquidity provision. By analyzing
multiple quality metrics across venues for a given security, we can rank them to identify
the venues with the best execution characteristics. This enables traders and market makers
to understand where their orders should be routed for optimal outcomes.

### Ranking Methodology
We rank venues through a multi-metric scoring approach:
1. **Collect daily venue quality metrics** across multiple dimensions (CBBO behavior,
   liquidity provision, fill characteristics)
2. **Normalize each metric independently** by ranking venues from best to worst per metric
3. **Aggregate rankings** by summing individual metric ranks for a composite score
4. **Final ranking** orders venues by composite score (lower rank = better overall venue)

This composite approach ensures venues that excel in one area but are weak in others
will have mid-range scores, reflecting balanced execution quality.

**Note on terminology:**
- **CBBO** (Consolidated Best Bid/Offer) = Best bid/offer across all venues (CBBO is the current standard term, replacing EBBO)
- **BBO** (Best Bid/Offer) = A single venue's best bid/offer (NOT consolidated)

### US Exchange Venues
We analyze execution across major US equity exchanges and alternative trading systems.
These market centers collectively capture nearly all US equity trading volume.

In [5]:
mics = [
    'XNAS',  # NASDAQ Stock Market
    'XBOS',  # Boston Stock Exchange
    'XPSX',  # Philadelphia Stock Exchange
    'XNYS',  # New York Stock Exchange
    'XASE',  # NYSE American (formerly AMEX)
    'ARCX',  # NYSE Arca
    'BATY',  # Bats BYX
    'BATS',  # Bats BZX
    'EDGA',  # EDGA
    'EDGX',  # EDGX
    'XCIS',  # NYSE Chicago
    'XCHI',  # Chicago Stock Exchange
    'IEXG',  # IEX
    'MEMX',  # MEMX
    'EPRL'   # Pearls
]

### Venue Quality Metrics
BMLL calculates daily venue quality metrics that quantify execution quality across multiple dimensions:

**Consolidated Best Bid/Offer (CBBO) Metrics:**
CBBO is the consolidated best bid/offer across all venues in a region. Note: CBBO is now the standard
term for what was previously called EBBO (European Best Bid Offer).
- *CBBOJoiner*: Orders that join the CBBO across all venues
- *CBBOSetter*: Orders that set the CBBO across all venues
- *TimeAtCBBO*: Duration orders spend at the CBBO
- *ExclusiveTimeAtCBBO*: Time this venue exclusively sets the CBBO

**Liquidity Metrics:**
- *FillProbability*: Likelihood a resting order gets filled (Ask1/Bid1 side)
- *CBBOMeanRestingTime*: Average time orders rest at CBBO before fill
- *TWALiquidityAroundCBBO*: Volume available near CBBO (time-weighted average)

**Trade Flow Metrics:**
- *TradeVolume*: Classification as Lit, Dark, Bilateral, or Non-standard
- *QuoteToTrade*: Relationship between quote updates and trades
- *SweepToFill*: Whether orders sweep across multiple price levels

**Pricing Metrics:**
- *Spread*: Bid-ask spread (both percentage and relative VWA)
- *CBBOOrderPercentage*: Percentage of orders contributing to CBBO

### Ranking Metrics
We select 21 specific metric+suffix combinations that provide comprehensive
execution quality coverage. Each is ranked independently, then summed for composite ranking.

### Metric Mapping to SQL Column Names

When running in Snowflake or Databricks, the ANALYTICS table uses SQL column names
instead of the BMLL API's metric|suffix notation. The mapping below translates
our 21 ranking metrics to their corresponding database columns.
This allows the same ranking algorithm to work across platforms with minimal changes.

In [6]:
LIQ_GROUP_BY_COLS_SF = {
    # CBBO Joiner metrics: Orders joining the CBBO (bid and ask sides, % and notional)
    # CBBO = Consolidated Best Bid/Offer (same as EBBO - now the standard term)
    'CBBOJoiner|AskTWAPercentage': 'CBBO_JOINER_ASK_TWA_PERCENTAGE',
    'CBBOJoiner|AskTWANotional': 'CBBO_JOINER_ASK_TWA_NOTIONAL',
    'CBBOJoiner|BidTWAPercentage': 'CBBO_JOINER_BID_TWA_PERCENTAGE',
    'CBBOJoiner|BidTWANotional': 'CBBO_JOINER_BID_TWA_NOTIONAL',
    # CBBO Setter metrics: Orders setting the CBBO (bid and ask sides, % and notional)
    'CBBOSetter|AskTWAPercentage': 'CBBO_SETTER_ASK_TWA_PERCENTAGE',
    'CBBOSetter|AskTWANotional': 'CBBO_SETTER_ASK_TWA_NOTIONAL',
    'CBBOSetter|BidTWAPercentage': 'CBBO_SETTER_BID_TWA_PERCENTAGE',
    'CBBOSetter|BidTWANotional': 'CBBO_SETTER_BID_TWA_NOTIONAL',
    # Exclusive time at CBBO: When this venue alone sets the CBBO
    'ExclusiveTimeAtCBBO|Percentage': 'EXCLUSIVE_TIME_AT_CBBO_PERCENTAGE',
    # Fill probability: Likelihood an order at the ask or bid gets filled
    'FillProbability|Ask1': 'FILL_PROBABILITY_ASK1',
    'FillProbability|Bid1': 'FILL_PROBABILITY_BID1',
    # Resting time at CBBO: How long orders rest at the consolidated best
    'CBBOMeanRestingTime|Filled': 'CBBO_MEAN_RESTING_TIME_FILLED',
    'CBBOOrderPercentage|Filled': 'CBBO_ORDER_PERCENTAGE_FILLED',
    # Time at CBBO: Total duration at the consolidated best bid/offer
    'TimeAtCBBO|Percentage': 'TIME_AT_CBBO_PERCENTAGE',
    # Spread relative to volume-weighted average price
    'Spread|RelVWA': 'SPREAD_BPS_VWA',
    # Trade volume by classification: lit vs dark vs bilateral vs non-standard execution
    'TradeVolume|Dark': 'TRADE_VOLUME_DARK',
    'TradeVolume|Lit': 'TRADE_VOLUME_LIT',
    'TradeVolume|Bi': 'TRADE_VOLUME_BI',
    'TradeVolume|Non': 'TRADE_VOLUME_NON',
    # Liquidity depth at 10bps around this venue's best bid/offer (BBO)
    # Note: BBO is this venue's best bid/offer, not the consolidated CBBO
    'TWALiquidityAroundBBO|Ask10bpsNotional': 'LIQUIDITY_AT_DEPTH_BBO_ASK10BPS_NOTIONAL',
    'TWALiquidityAroundBBO|Bid10bpsNotional': 'LIQUIDITY_AT_DEPTH_BBO_BID10BPS_NOTIONAL',
    # Sweep-to-fill: Spread when orders sweep across multiple price levels at 100k USD size
    'SweepToFill|100kUSDTWASpread': 'SWEEP_TO_FILL100K_USD_SPREAD',
}

## Part 2: Ranking Implementation Across Platforms

The `get_ranking()` function implements our multi-metric ranking algorithm across all platforms:

**Step 1: Fetch metrics from BMLL**
- Query all configured metrics across the specified date range and MIC list
- Filter to daily frequency metrics

**Step 2: Aggregate by venue**
- Group data by ListingId (which maps 1:1 to venues/MICs)
- Take mean of each metric across the date range
- This produces a single row per venue with 21 metric values

**Step 3: Rank metrics independently**
- For each metric, rank venues from 1 (best) to N (worst)
- Higher metric values = better, so use descending=False
- NaN values ranked lowest (bottom)

**Step 4: Composite scoring**
- Sum the 21 individual metric ranks per venue
- Lower RankSum = better overall venue

**Step 5: Final ranking**
- Rank venues by RankSum (1 = best, N = worst)
- Add venue identifiers (MIC, Ticker, dates) for context

The result is a ranked table showing each venue's position,
composite score, and date range for reproducibility.

### Snowflake Implementation

For Snowflake, we run the same ranking algorithm directly on the ANALYTICS table
using SQL aggregations and window functions. This is more efficient than fetching
data and processing in Python, and leverages Snowflake's MPP compute.

In [7]:
def get_ranking(ticker, sd, ed):
    """
    Snowflake implementation: Rank venues using SQL aggregations on ANALYTICS table.

    Fetches pre-computed venue quality metrics from the ANALYTICS table and computes
    rankings using Snowflake's aggregation and window functions.

    Parameters:
        ticker (str): Stock ticker symbol
        sd (str): Start date in YYYY-MM-DD format
        ed (str): End date in YYYY-MM-DD format

    Returns:
        pd.DataFrame: Same schema as notebook version:
                     [ListingId, MIC, Rank, RankSum, Ticker, StartDate, EndDate]

    Note:
        Requires an active Snowpark session named 'session' in the Snowflake environment.
        Metric columns are referenced by SQL names (e.g., CBBO_JOINER_ASK_TWA_PERCENTAGE)
        rather than the BMLL API notation.
    """
    # Step 1: Build SQL column list from metric mapping
    cols = list(LIQ_GROUP_BY_COLS_SF.values())
    cols_sql = ", ".join(cols)
    mic_list = ", ".join(f"'{m}'" for m in mics)

    # Step 2: Query ANALYTICS table for the date range and venues
    sql = f"""
        SELECT
            LISTING_ID AS "ListingId",
            MIC,
            {cols_sql}
        FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."ANALYTICS"
        WHERE TICKER = '{ticker}'
          AND MIC IN ({mic_list})
          AND TRADING_DATE BETWEEN '{sd}' AND '{ed}'
    """
    liq = session.sql(sql).to_pandas()

    # Handle empty result - return as-is if no data
    if liq.empty:
        return liq

    # Step 3: Convert volume columns to float for aggregation
    volume_cols = ['TRADE_VOLUME_DARK', 'TRADE_VOLUME_LIT', 'TRADE_VOLUME_BI', 'TRADE_VOLUME_NON']
    for col in volume_cols:
        if col in liq.columns:
            liq[col] = liq[col].astype('float64')

    # Step 4: Aggregate metrics by venue (mean across date range)
    liq_gb = liq.groupby(['ListingId'], as_index=False)[cols].mean()

    # Step 5: Rank each metric independently
    for m in cols:
        liq_gb[f'{m}Rank'] = liq_gb[m].rank(ascending=False, na_option='bottom')

    # Step 6: Sum ranks and compute final ranking
    liq_gb['RankSum'] = liq_gb.iloc[:, -len(cols):].sum(axis=1)
    liq_gb['Rank'] = liq_gb['RankSum'].rank(ascending=True, na_option='bottom')

    # Step 7: Format output with MIC and context
    rank = liq_gb[['ListingId', 'Rank', 'RankSum']].sort_values('RankSum')
    rank = rank.merge(liq[['ListingId', 'MIC']].drop_duplicates(), on='ListingId', how='left')
    rank['Ticker'] = ticker
    rank['StartDate'] = sd
    rank['EndDate'] = ed

    return rank

## Part 3: Single-Ticker Venue Ranking Analysis

Demonstrate venue ranking for a single security to identify the best execution venues.
We analyze Tesla (TSLA) across US venues for the first half of 2023.

**Output columns:**
- **Rank**: Final composite ranking (1 = best venue)
- **RankSum**: Sum of individual metric ranks (lower = better)
- **MIC**: Market Identifier Code (venue/exchange name)
- **ListingId**: BMLL's unique listing identifier
- **Ticker, StartDate, EndDate**: Query parameters for reference

In [8]:
print(f"\n{'='*80}")
print("SINGLE-TICKER VENUE RANKING: TESLA (TSLA)")
print(f"{'='*80}\n")

rankings = get_ranking('TSLA', '2023-01-01', '2023-06-21')

if len(rankings) > 0:
    print(f"Analyzed {len(rankings)} venues over 6-month period\n")

    # Show top 10 venues
    print(rankings[['MIC', 'Rank', 'RankSum']].head(10).to_string(index=False))

    print(f"\n{'='*80}\n")
    print(f"BEST VENUE: {rankings.iloc[0]['MIC']} (Rank {int(rankings.iloc[0]['Rank'])}, Score {rankings.iloc[0]['RankSum']:.1f})")
    print(f"{'='*80}\n")


SINGLE-TICKER VENUE RANKING: TESLA (TSLA)



Analyzed 15 venues over 6-month period

 MIC  Rank  RankSum
EDGX   1.0    118.0
XNAS   2.0    122.0
ARCX   3.0    142.0
BATS   4.0    146.0
MEMX   5.0    155.0
XNYS   6.0    157.0
XCIS   7.0    184.0
XPSX   8.0    185.0
XASE   9.0    189.0
XCHI  10.0    195.0


BEST VENUE: EDGX (Rank 1, Score 118.0)



## Part 4: Multi-Ticker Comparative Analysis

Extend the analysis across multiple securities to identify venue consistency.
This helps determine which venues are reliably high-quality across the market.

In [9]:
print(f"\n{'='*80}")
print("MULTI-TICKER COMPARATIVE ANALYSIS")
print(f"{'='*80}\n")

# Analyze multiple major stocks
analysis_tickers = ['AAPL', 'MSFT', 'GOOGL', 'TSLA']
comparison_results = []

for ticker in analysis_tickers:
    print(f"Ranking venues for {ticker}...")
    ticker_rankings = get_ranking(ticker, '2023-01-01', '2023-06-21')

    if len(ticker_rankings) > 0:
        # Get top 5 venues for this ticker
        top_5 = ticker_rankings.head(5)[['MIC', 'Rank', 'RankSum']].copy()
        top_5['Ticker'] = ticker
        comparison_results.append(top_5)

# Combine and display results
if comparison_results:
    print(f"\n{'='*80}")
    print("TOP 5 VENUES BY TICKER")
    print(f"{'='*80}\n")

    for ticker in analysis_tickers:
        ticker_data = next((df for df in comparison_results if (df['Ticker'] == ticker).any()), None)
        if ticker_data is not None:
            print(f"\n{ticker}:")
            print(ticker_data[['MIC', 'Rank', 'RankSum']].to_string(index=False))

    print(f"\n{'='*80}\n")

    # Identify most consistent (frequently appearing) venues
    all_comparison = pd.concat(comparison_results, ignore_index=True)
    venue_appearances = all_comparison['MIC'].value_counts().reset_index()
    venue_appearances.columns = ['MIC', 'Appearances_in_Top5']

    print("VENUE CONSISTENCY - Appearances in Top 5 Across Tickers:")
    print(venue_appearances.to_string(index=False))
    print(f"\n{'='*80}\n")


MULTI-TICKER COMPARATIVE ANALYSIS

Ranking venues for AAPL...


Ranking venues for MSFT...


Ranking venues for GOOGL...


Ranking venues for TSLA...



TOP 5 VENUES BY TICKER


AAPL:
 MIC  Rank  RankSum
ARCX   1.0    106.0
XNAS   2.0    107.0
EDGX   3.0    124.0
BATS   4.0    125.0
XNYS   5.0    160.0

MSFT:
 MIC  Rank  RankSum
XNAS   1.0    122.0
ARCX   2.0    128.0
MEMX   3.0    139.0
EDGX   4.0    142.0
BATS   5.0    147.0

GOOGL:
 MIC  Rank  RankSum
ARCX   1.0    107.0
XNAS   2.0    119.0
EDGX   3.0    121.0
BATS   4.0    136.0
EPRL   5.0    145.0

TSLA:
 MIC  Rank  RankSum
EDGX   1.0    118.0
XNAS   2.0    122.0
ARCX   3.0    142.0
BATS   4.0    146.0
MEMX   5.0    155.0


VENUE CONSISTENCY - Appearances in Top 5 Across Tickers:
 MIC  Appearances_in_Top5
ARCX                    4
XNAS                    4
EDGX                    4
BATS                    4
MEMX                    2
XNYS                    1
EPRL                    1


